# Treasure Trail — SBC Kids' Bible SLM
### QLoRA fine-tune + base-vs-tuned eval

**First: Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Flow: clone → install → **baseline eval** (base *flattens/caves*) → **QLoRA fine-tune** (1,095-record dataset) → **tuned eval** → **base-vs-tuned results table**.

The eval judge runs through your **TrueFoundry** gateway. Training + generation need **no key** — they run free on the T4.

In [ ]:
!git clone https://github.com/graceyan212/bible-slm.git
%cd bible-slm
!pip install -q unsloth anthropic openai

## 1 · Config + judge key

**Your TrueFoundry `user-…` key goes here.** Two safe ways (never put it in the repo):

- **Set-once (recommended):** click the **🔑 key icon** in Colab's left sidebar → **+ Add new secret** → Name `TRUEFOUNDRY_API_KEY`, Value = your `user-…` key → toggle **Notebook access ON**. Then just Run all — no prompt.
- **Or:** leave secrets empty and the cell will **prompt** you to paste it (it's never saved to the notebook).

Also set `JUDGE_BASE_URL` and `JUDGE_MODEL` to match your console (TrueFoundry → **LLM Playground → Code Snippets**).

In [ ]:
import os, getpass

# --- Base model to fine-tune (free on the T4; no key needed) ---
MODEL = "unsloth/Qwen3-4B-Instruct-2507"        # faster/lighter: "unsloth/Qwen3-1.7B-Instruct"
os.environ["BASE_MODEL"] = MODEL

# --- Judge (eval scoring) via TrueFoundry gateway ---
os.environ["JUDGE_BASE_URL"] = "https://gateway.truefoundry.ai"            # or your org's .../api/llm/api/inference/openai
os.environ["JUDGE_MODEL"]    = "anthropic-main/claude-3-5-sonnet-20241022"  # <- exact provider-account/model from YOUR console

# API key: Colab Secret 'TRUEFOUNDRY_API_KEY' if present, else prompt (paste your user-... key).
key = None
try:
    from google.colab import userdata
    key = userdata.get('TRUEFOUNDRY_API_KEY')
except Exception:
    pass
os.environ["JUDGE_API_KEY"] = key or getpass.getpass('TrueFoundry API key (user-...): ')

# (Prefer a raw Anthropic key instead? Comment JUDGE_* above and uncomment:)
# os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')  # or getpass.getpass('Anthropic key: ')

print('Base:', MODEL, '| judge:', os.environ['JUDGE_MODEL'], '| key loaded:', bool(os.environ['JUDGE_API_KEY']))

## 2 · Baseline eval — run BEFORE training
Proves the delta target exists: expect the base to **flatten** on baptism/eternal-security and **cave** under pushback.

In [ ]:
!python eval/run_eval.py --model base --hf {MODEL} --out results_base.json

## 3 · Fine-tune (QLoRA, ~30–60 min)
Trains on `data/train_v2.jsonl` (1,095 verified records; prints data hash) → writes `./sbc-lora`.

In [ ]:
!python train/train_qlora.py

## 4 · Tuned eval — same 52 scenarios, same system prompt

In [ ]:
!python eval/run_eval.py --model tuned --hf {MODEL} --adapter ./sbc-lora --out results_tuned.json

## 5 · Results table — base vs tuned (headline artifact)

In [ ]:
!python eval/run_eval.py --compare results_base.json results_tuned.json --md results_table.md
from IPython.display import Markdown, display
display(Markdown(open('results_table.md').read()))

**A win =** tuned beats base on **demo FLATTEN rate** + **hold-under-pressure**, without open OVER_HOLD / deflect-leak rising or safe_core task-quality regressing.

Colab wipes on disconnect — the next cell downloads your results. Send `results_table.md` back and I'll build the demo.

In [ ]:
from google.colab import files
for f in ['results_table.md', 'results_base.json', 'results_tuned.json']:
    files.download(f)